# Heatwave Analysis — Dhaka, Bangladesh (1972–2024)

End-to-end pipeline producing all journal-quality figures.
Run all cells top-to-bottom. Outputs saved to `figures/`.

In [ ]:
"""
Heatwave Analysis — Dhaka, Bangladesh (1972–2024)
=================================================
Produces all journal-quality figures for the manuscript.
Run:  python analysis.py
Output: figures/*.png  (300 DPI, journal-ready)
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import scipy.stats as stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.stats.stattools import durbin_watson
import os

In [ ]:
# ── OUTPUT ─────────────────────────────────────────────────────────────────────
OUT = "figures"
os.makedirs(OUT, exist_ok=True)

In [ ]:
# ── JOURNAL STYLE ──────────────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica Neue", "DejaVu Sans"],
    "font.size":          9,
    "axes.titlesize":     10,
    "axes.titleweight":   "bold",
    "axes.labelsize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "legend.fontsize":    8,
    "legend.frameon":     True,
    "legend.framealpha":  0.85,
    "legend.edgecolor":   "0.8",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.8,
    "xtick.major.width":  0.8,
    "ytick.major.width":  0.8,
    "grid.alpha":         0.35,
    "grid.linewidth":     0.5,
    "grid.color":         "0.7",
    "lines.linewidth":    1.6,
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
})

# Colorblind-safe palette (Wong 2011)
C = {
    "blue":    "#0072B2",
    "orange":  "#E69F00",
    "red":     "#D55E00",
    "green":   "#009E73",
    "sky":     "#56B4E9",
    "yellow":  "#F0E442",
    "purple":  "#CC79A7",
    "black":   "#000000",
    "gray":    "#808080",
    "lgray":   "#CCCCCC",
}

SAVE_KW = dict(dpi=300, bbox_inches="tight", pad_inches=0.05)


def savefig(name):
    path = os.path.join(OUT, name)
    plt.savefig(path, **SAVE_KW)
    plt.close()
    print(f"  Saved → {path}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════
print("Loading data …")

df = pd.read_csv("data/1972_2024_Heatwave_Daily.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
df = df.set_index("timestamp")
df.index.name = "date"

# Rename columns to readable names
# Pattern: original = max, .1 = min, .2 = mean  (confirmed from data inspection)
df.rename(columns={
    "Dhaka Temperature [2 m elevation corrected]":    "tmax",
    "Dhaka Temperature [2 m elevation corrected].1":  "tmin",
    "Dhaka Temperature [2 m elevation corrected].2":  "tmean",
    "Dhaka Precipitation Total":                      "precip",
    "Dhaka Relative Humidity [2 m]":                  "rh_max",
    "Dhaka Relative Humidity [2 m].1":                "rh_min",
    "Dhaka Relative Humidity [2 m].2":                "rh_mean",
    "Dhaka Wind Gust":                                "wind_gust_max",
    "Dhaka Wind Gust.1":                              "wind_gust_min",
    "Dhaka Wind Gust.2":                              "wind_gust_mean",
    "Dhaka Wind Speed [10 m]":                        "wind_max",
    "Dhaka Wind Speed [10 m].1":                      "wind_min",
    "Dhaka Cloud Cover Total":                        "cloud",
    "Dhaka Sunshine Duration":                        "sunshine",
    "Dhaka Shortwave Radiation":                      "sw_rad",
    "Dhaka Longwave Radiation":                       "lw_rad",
    "Dhaka UV Radiation":                             "uv",
    "Dhaka Direct Shortwave Radiation":               "direct_sw",
    "Dhaka Mean Sea Level Pressure [MSL]":            "mslp_max",
    "Dhaka Mean Sea Level Pressure [MSL].1":          "mslp_min",
    "Dhaka Mean Sea Level Pressure [MSL].2":          "mslp_mean",
    "Dhaka Evapotranspiration":                       "et",
    "Dhaka Vapor Pressure Deficit [2 m]":             "vpd_max",
    "Dhaka Vapor Pressure Deficit [2 m].1":           "vpd_min",
    "Dhaka Vapor Pressure Deficit [2 m].2":           "vpd_mean",
    "Dhaka Soil Temperature [0-7 cm down]":           "soil_temp_max",
    "Dhaka Soil Temperature [0-7 cm down].1":         "soil_temp_min",
    "Dhaka Soil Temperature [0-7 cm down].2":         "soil_temp_mean",
    "Dhaka Soil Moisture [0-7 cm down]":              "sm_max",
    "Dhaka Soil Moisture [0-7 cm down].1":            "sm_min",
    "Dhaka Soil Moisture [0-7 cm down].2":            "sm_mean",
}, inplace=True)

In [ ]:
# ── GFW deforestation ──────────────────────────────────────────────────────────
gfw_raw = pd.read_csv("data/GFW_Dhaka.csv")
gfw = (gfw_raw[["Tree_Cover_Loss_Year", "umd_tree_cover_loss__ha"]]
       .dropna(subset=["Tree_Cover_Loss_Year"])
       .groupby("Tree_Cover_Loss_Year", as_index=False)
       .sum()
       .rename(columns={"Tree_Cover_Loss_Year": "year",
                         "umd_tree_cover_loss__ha": "tree_loss_ha"}))
gfw["year"] = gfw["year"].astype(int)

print(f"  Daily records : {len(df):,}  ({df.index.min().date()} → {df.index.max().date()})")
print(f"  Missing tmax  : {df['tmax'].isna().sum()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. DERIVED VARIABLES
# ══════════════════════════════════════════════════════════════════════════════

# 2a. Heat Index (NWS Rothfusz regression, valid when T ≥ 27°C and RH ≥ 40%)
#     Inputs: Tmax (°C), RH_mean (%)
def heat_index_celsius(T, RH):
    """NWS Rothfusz Heat Index, returned in °C. NaN for T < 27°C or RH < 40%."""
    hi = np.full_like(T, np.nan, dtype=float)
    mask = (T >= 27) & (RH >= 40)
    t = T[mask]
    r = RH[mask]
    hi[mask] = (
        -8.78469475556
        + 1.61139411   * t
        + 2.33854883889 * r
        - 0.14611605   * t * r
        - 0.012308094  * t**2
        - 0.0164248277778 * r**2
        + 0.002211732  * t**2 * r
        + 0.00072546   * t * r**2
        - 0.000003582  * t**2 * r**2
    )
    return hi

df["heat_index"] = heat_index_celsius(df["tmax"].values, df["rh_mean"].values)

# Heat index is only computed for Tmax ≥ 27°C and RH ≥ 40°.
# Because Dhaka's baseline humidity is persistently high, we do NOT use a
# fixed HI threshold for heatwave counting (it would flag most days).
# Instead we use two meaningful metrics:
#   (a) HI on actual heatwave days (Tmax ≥ 36°C) — how bad does a heatwave feel?
#   (b) Annual mean HI trend vs Tmax trend — is humidity-amplified heat rising?
#   (c) HI percentile threshold: days exceeding the historical 95th-percentile of HI
HI_95 = df["heat_index"].quantile(0.95)
df["hw_day_hi95"] = (df["heat_index"] >= HI_95).fillna(0).astype(int)

# 2b. Nighttime recovery gap  (Tmax − Tmin)
df["recovery_gap"] = df["tmax"] - df["tmin"]

# 2c. Soil moisture percentile rank (within calendar month, across all years)
df["sm_month"] = df.index.month
df["sm_pct"] = df.groupby("sm_month")["sm_mean"].transform(
    lambda x: x.rank(pct=True) * 100
)
df.drop(columns="sm_month", inplace=True)

# 2d. Heatwave flag — Tmax ≥ 36°C
HW_THRESH = 36.0
df["hw_day"]    = (df["tmax"] >= HW_THRESH).astype(int)
df["hw_day_hi"] = (df["heat_index"] >= HW_THRESH).fillna(0).astype(int)

# 2e. Compound event: hw_day AND soil moisture in bottom 25th percentile of month
df["compound"] = ((df["hw_day"] == 1) & (df["sm_pct"] <= 25)).astype(int)

In [ ]:
# ── Annual aggregates ──────────────────────────────────────────────────────────
ann = df.resample("YE").agg(
    tmax_mean   = ("tmax",        "mean"),
    tmax_max    = ("tmax",        "max"),
    tmin_mean   = ("tmin",        "mean"),
    tmean_mean  = ("tmean",       "mean"),
    hw_days     = ("hw_day",      "sum"),
    hw_days_hi95 = ("hw_day_hi95", "sum"),
    gap_mean    = ("recovery_gap","mean"),
    compound    = ("compound",    "sum"),
    vpd_mean    = ("vpd_mean",    "mean"),
    sm_mean     = ("sm_mean",     "mean"),
).copy()
ann.index = ann.index.year

# Merge deforestation
ann = ann.merge(gfw.set_index("year")[["tree_loss_ha"]], left_index=True,
                right_index=True, how="left")

# Linear trend helper
def lin_trend(x, y):
    mask = ~np.isnan(y)
    slope, intercept, r, p, se = stats.linregress(x[mask], y[mask])
    return slope, intercept, r**2, p

yrs = ann.index.values.astype(float)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. FIGURES
# ══════════════════════════════════════════════════════════════════════════════

## Figure 1 — Overview: Warming, Heatwave Days, Deforestation

In [ ]:
# ── Figure 1: Overview — Warming, Heatwave Days, Deforestation ────────────────
print("Figure 1: Overview …")

fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.8))

# Panel A — Annual mean temperature with trend
ax = axes[0]
sl, ic, r2, p = lin_trend(yrs, ann["tmax_mean"].values)
ax.bar(ann.index, ann["tmax_mean"], color=C["sky"], width=0.8, alpha=0.7,
       label="Annual mean Tmax")
trend_y = sl * yrs + ic
ax.plot(ann.index, trend_y, color=C["red"], lw=1.8,
        label=f"Trend: +{sl*10:.3f} °C/decade")
ax.set_xlabel("Year")
ax.set_ylabel("Mean Tmax (°C)")
ax.set_title("A  Warming Trend")
ax.legend(loc="upper left", fontsize=7)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))
ax.set_xlim(1971, 2025)

# Panel B — Annual heatwave days
ax = axes[1]
colors_hw = [C["orange"] if y >= 2000 else C["sky"] for y in ann.index]
ax.bar(ann.index, ann["hw_days"], color=colors_hw, width=0.8, alpha=0.85)
roll5 = ann["hw_days"].rolling(5, center=True).mean()
ax.plot(ann.index, roll5, color=C["red"], lw=1.8, label="5-yr rolling mean")
ax.axhline(ann["hw_days"].mean(), color=C["gray"], lw=1, ls="--",
           label=f"Mean: {ann['hw_days'].mean():.1f} days")
ax.set_xlabel("Year")
ax.set_ylabel("Heatwave days (Tmax ≥ 36 °C)")
ax.set_title("B  Annual Heatwave Days")
ax.legend(loc="upper left", fontsize=7)
ax.set_xlim(1971, 2025)

# Custom legend patch for pre/post 2000
from matplotlib.patches import Patch
leg_els = [Patch(fc=C["sky"], label="1972–1999"),
           Patch(fc=C["orange"], label="2000–2024"),
           Line2D([0],[0], color=C["red"], lw=1.8, label="5-yr mean"),
           Line2D([0],[0], color=C["gray"], lw=1, ls="--", label="Historical mean")]
axes[1].legend(handles=leg_els, fontsize=7, loc="upper left")

# Panel C — Deforestation
ax = axes[2]
gfw_plot = gfw[(gfw["year"] >= 2001) & (gfw["year"] <= 2023)]
ax.bar(gfw_plot["year"], gfw_plot["tree_loss_ha"] / 1000, color=C["green"],
       width=0.8, alpha=0.8)
cumulative = gfw_plot["tree_loss_ha"].cumsum() / 1000
ax2 = ax.twinx()
ax2.plot(gfw_plot["year"], cumulative, color=C["red"], lw=1.8, label="Cumulative")
ax2.set_ylabel("Cumulative loss (×10³ ha)", color=C["red"], fontsize=8)
ax2.tick_params(axis="y", labelcolor=C["red"], labelsize=8)
ax2.spines["top"].set_visible(False)
ax.set_xlabel("Year")
ax.set_ylabel("Annual loss (×10³ ha)")
ax.set_title("C  Tree-Cover Loss (Dhaka)")

fig.tight_layout(w_pad=2.5)
savefig("fig1_overview.png")

## Figure 2 — Temperature Structure

In [ ]:
# ── Figure 2: Temperature Structure ───────────────────────────────────────────
print("Figure 2: Temperature structure …")

fig, axes = plt.subplots(2, 2, figsize=(7.5, 5.5))

# A — Daily temperature 2020–2024
ax = axes[0, 0]
recent = df.loc["2020":"2024", ["tmax", "tmin"]]
ax.fill_between(recent.index, recent["tmin"], recent["tmax"],
                color=C["sky"], alpha=0.5, label="Tmin–Tmax range")
ax.plot(recent.index, recent["tmax"], color=C["red"], lw=0.6, alpha=0.8)
ax.plot(recent.index, recent["tmin"], color=C["blue"], lw=0.6, alpha=0.8)
ax.axhline(HW_THRESH, color=C["orange"], lw=1, ls="--",
           label=f"Heatwave threshold ({HW_THRESH} °C)")
ax.set_xlabel("Year")
ax.set_ylabel("Temperature (°C)")
ax.set_title("A  Daily Temperature Range (2020–2024)")
ax.legend(fontsize=7)

# B — Annual max temperature trend
ax = axes[0, 1]
sl2, ic2, r2b, p2 = lin_trend(yrs, ann["tmax_max"].values)
ax.scatter(ann.index, ann["tmax_max"], s=18, color=C["blue"], alpha=0.7,
           label="Annual Tmax")
ax.plot(ann.index, sl2 * yrs + ic2, color=C["red"], lw=1.8,
        label=f"Trend: +{sl2*10:.3f} °C/decade\n(R²={r2b:.2f}, p={p2:.3f})")
ax.set_ylabel("Annual maximum Tmax (°C)")
ax.set_title("B  Trend in Annual Tmax")
ax.legend(fontsize=7)
ax.set_xlim(1971, 2025)

# C — Seasonal climatology
ax = axes[1, 0]
monthly_mean = df.groupby(df.index.month)["tmax"].mean()
monthly_std  = df.groupby(df.index.month)["tmax"].std()
months = np.arange(1, 13)
month_labels = ["J","F","M","A","M","J","J","A","S","O","N","D"]
ax.bar(months, monthly_mean, color=C["orange"], alpha=0.8, width=0.7,
       label="Mean Tmax")
ax.errorbar(months, monthly_mean, yerr=monthly_std, fmt="none",
            color=C["black"], capsize=3, lw=1)
ax.axhline(HW_THRESH, color=C["red"], lw=1, ls="--",
           label=f"{HW_THRESH} °C threshold")
ax.set_xticks(months)
ax.set_xticklabels(month_labels)
ax.set_ylabel("Tmax (°C)")
ax.set_title("C  Seasonal Climatology (1972–2024)")
ax.legend(fontsize=7)

# D — Decadal box plots
ax = axes[1, 1]
df["decade"] = (df.index.year // 10) * 10
decade_data = [df.loc[df["decade"] == d, "tmax"].dropna().values
               for d in sorted(df["decade"].unique())]
decade_labels = [f"{d}s" for d in sorted(df["decade"].unique())]
bp = ax.boxplot(decade_data, patch_artist=True, medianprops=dict(color="white", lw=2),
                flierprops=dict(marker=".", markersize=2, alpha=0.3),
                whiskerprops=dict(lw=0.8), capprops=dict(lw=0.8))
colors_dec = plt.cm.RdYlBu_r(np.linspace(0.2, 0.9, len(decade_data)))
for patch, c in zip(bp["boxes"], colors_dec):
    patch.set_facecolor(c)
    patch.set_alpha(0.8)
ax.set_xticklabels(decade_labels, fontsize=8)
ax.axhline(HW_THRESH, color=C["red"], lw=1, ls="--", label=f"{HW_THRESH} °C")
ax.set_ylabel("Daily Tmax (°C)")
ax.set_title("D  Decadal Tmax Distribution")
ax.legend(fontsize=7)
df.drop(columns="decade", inplace=True)

fig.tight_layout(w_pad=2.5, h_pad=3)
savefig("fig2_temperature_structure.png")

## Figure 3 — Heatwave Characteristics

In [ ]:
# ── Figure 3: Heatwave Characteristics ────────────────────────────────────────
print("Figure 3: Heatwave characteristics …")

# Build event-level summary
hw_events = []
in_event, start, length, intensities = False, None, 0, []
for date, row in df.iterrows():
    if row["hw_day"] == 1:
        if not in_event:
            in_event, start, length, intensities = True, date, 0, []
        length += 1
        intensities.append(row["tmax"])
    else:
        if in_event:
            hw_events.append({"start": start, "length": length,
                               "peak": max(intensities),
                               "mean": np.mean(intensities),
                               "year": start.year,
                               "month": start.month})
        in_event = False
if in_event:
    hw_events.append({"start": start, "length": length, "peak": max(intensities),
                      "mean": np.mean(intensities), "year": start.year,
                      "month": start.month})
hw_ev = pd.DataFrame(hw_events)

fig, axes = plt.subplots(2, 2, figsize=(7.5, 5.5))

# A — Annual heatwave days + rolling mean
ax = axes[0, 0]
ax.bar(ann.index, ann["hw_days"], color=C["sky"], width=0.8, alpha=0.75)
roll5 = ann["hw_days"].rolling(5, center=True).mean()
ax.plot(ann.index, roll5, color=C["red"], lw=1.8, label="5-yr rolling mean")
ax.axhline(ann["hw_days"].mean(), color=C["gray"], ls="--", lw=1,
           label=f"Mean = {ann['hw_days'].mean():.1f} d/yr")
sl_hw, ic_hw, r2_hw, p_hw = lin_trend(yrs, ann["hw_days"].values)
ax.set_xlabel("Year"); ax.set_ylabel("Heatwave days")
ax.set_title("A  Annual Heatwave Days (Tmax ≥ 36 °C)")
ax.legend(fontsize=7)

# B — Monthly distribution
ax = axes[0, 1]
monthly_hw = df.groupby(df.index.month)["hw_day"].sum()
bar_colors = [C["red"] if m in [4, 5] else C["orange"] if m in [3, 6] else C["sky"]
              for m in range(1, 13)]
ax.bar(range(1, 13), monthly_hw, color=bar_colors, width=0.75, alpha=0.85)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax.set_xlabel("Month"); ax.set_ylabel("Total heatwave days (1972–2024)")
ax.set_title("B  Monthly Distribution of Heatwave Days")
legend_els = [Patch(fc=C["red"], label="Peak (Apr–May)"),
              Patch(fc=C["orange"], label="Shoulder (Mar, Jun)"),
              Patch(fc=C["sky"], label="Other")]
ax.legend(handles=legend_els, fontsize=7)

# C — Year-month heatmap
ax = axes[1, 0]
hw_pivot = df.pivot_table(index=df.index.year, columns=df.index.month,
                          values="hw_day", aggfunc="sum").fillna(0)
hw_pivot.columns = ["J","F","M","A","M","J","J","A","S","O","N","D"]
im = ax.imshow(hw_pivot.values, aspect="auto", cmap="YlOrRd",
               interpolation="nearest", vmin=0, vmax=20)
ax.set_yticks(range(0, len(hw_pivot), 10))
ax.set_yticklabels(hw_pivot.index[::10], fontsize=7)
ax.set_xticks(range(12))
ax.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax.set_xlabel("Month"); ax.set_ylabel("Year")
ax.set_title("C  Year–Month Heatwave Heatmap")
plt.colorbar(im, ax=ax, label="Days / month", shrink=0.8, pad=0.02)

# D — Event duration histogram
ax = axes[1, 1]
durations = hw_ev["length"].values
bins = np.arange(0.5, max(durations) + 1.5, 1)
ax.hist(durations, bins=bins, color=C["orange"], edgecolor="white",
        lw=0.5, alpha=0.85)
ax.set_xlabel("Event duration (days)")
ax.set_ylabel("Number of events")
ax.set_title(f"D  Event Duration Distribution (n={len(hw_ev)})")
med_d = np.median(durations)
ax.axvline(med_d, color=C["red"], lw=1.5, ls="--",
           label=f"Median = {med_d:.0f} d")
ax.legend(fontsize=7)

fig.tight_layout(w_pad=2.5, h_pad=3)
savefig("fig3_heatwave_characteristics.png")

## Figure 4 — Heat Index (Apparent Temperature)

In [ ]:
# ── Figure 4: Heat Index — Apparent Temperature ───────────────────────────────
print("Figure 4: Heat index analysis …")

fig, axes = plt.subplots(1, 3, figsize=(7.5, 3.0))

# A — Annual mean Heat Index vs annual mean Tmax (trend comparison)
ax = axes[0]
ann_hi_yr  = df.groupby(df.index.year)["heat_index"].mean()
ann_tx_yr  = df.groupby(df.index.year)["tmax"].mean()
sl_hi, ic_hi, r2_hi, p_hi = lin_trend(ann_hi_yr.index.values.astype(float),
                                        ann_hi_yr.values)
sl_tx2, ic_tx2, _, _ = lin_trend(ann_tx_yr.index.values.astype(float),
                                   ann_tx_yr.values)
ax.plot(ann_hi_yr.index, ann_hi_yr.values, color=C["red"], lw=1.5,
        alpha=0.8, label=f"Heat Index (+{sl_hi*10:.3f} °C/dec)")
ax.plot(ann_tx_yr.index, ann_tx_yr.values, color=C["blue"], lw=1.5,
        alpha=0.8, label=f"Tmax (+{sl_tx2*10:.3f} °C/dec)")
ax.plot(ann_hi_yr.index, sl_hi * ann_hi_yr.index + ic_hi,
        color=C["red"], lw=1, ls="--", alpha=0.6)
ax.plot(ann_tx_yr.index, sl_tx2 * ann_tx_yr.index + ic_tx2,
        color=C["blue"], lw=1, ls="--", alpha=0.6)
ax.set_xlabel("Year")
ax.set_ylabel("Temperature (°C)")
ax.set_title("A  Heat Index vs Tmax Trend")
ax.legend(fontsize=7)

# B — Extra heat burden on heatwave days: HI minus Tmax
ax = axes[1]
hw_only = df[df["hw_day"] == 1].dropna(subset=["heat_index"])
burden = hw_only["heat_index"] - hw_only["tmax"]
burden_ann = burden.groupby(burden.index.year).mean()
sl_b, ic_b, r2_b, p_b = lin_trend(burden_ann.index.values.astype(float),
                                    burden_ann.values)
ax.bar(burden_ann.index, burden_ann.values, color=C["orange"], width=0.8,
       alpha=0.8)
ax.plot(burden_ann.index, sl_b * burden_ann.index + ic_b,
        color=C["red"], lw=1.8,
        label=f"Trend: {sl_b*10:+.3f} °C/decade (p={p_b:.3f})")
ax.set_xlabel("Year")
ax.set_ylabel("Heat Index − Tmax (°C)")
ax.set_title("B  Humidity Heat Burden on Heatwave Days")
ax.legend(fontsize=7)

# C — Annual days exceeding historical 95th-percentile of Heat Index
ax = axes[2]
sl_95, ic_95, r2_95, p_95 = lin_trend(yrs, ann["hw_days_hi95"].values)
ax.bar(ann.index, ann["hw_days_hi95"], color=C["purple"], width=0.8, alpha=0.75)
ax.plot(ann.index, ann["hw_days_hi95"].rolling(5, center=True).mean(),
        color=C["black"], lw=1.8, label="5-yr mean")
ax.plot(ann.index, sl_95 * yrs + ic_95, color=C["red"], lw=1.5, ls="--",
        label=f"Trend p={p_95:.3f}")
ax.set_xlabel("Year")
ax.set_ylabel("Days exceeding HI 95th percentile")
ax.set_title(f"C  Extreme Heat Days (HI ≥ 95th pct.)")
ax.legend(fontsize=7)

fig.tight_layout(w_pad=2.5)
savefig("fig4_heat_index.png")

## Figure 5 — Nighttime Recovery Gap

In [ ]:
# ── Figure 5: Nighttime Recovery Gap ──────────────────────────────────────────
print("Figure 5: Nighttime recovery gap …")

fig, axes = plt.subplots(1, 3, figsize=(7.5, 3.0))

# A — Annual mean recovery gap with trend
ax = axes[0]
sl_g, ic_g, r2_g, p_g = lin_trend(yrs, ann["gap_mean"].values)
ax.scatter(ann.index, ann["gap_mean"], s=20, color=C["blue"], alpha=0.7)
ax.plot(ann.index, sl_g * yrs + ic_g, color=C["red"], lw=1.8,
        label=f"Trend: {sl_g*10:+.3f} °C/decade\n(R²={r2_g:.2f}, p={p_g:.3f})")
ax.fill_between(ann.index,
                ann["gap_mean"].rolling(5, center=True).mean() - ann["gap_mean"].rolling(5, center=True).std(),
                ann["gap_mean"].rolling(5, center=True).mean() + ann["gap_mean"].rolling(5, center=True).std(),
                alpha=0.15, color=C["blue"])
ax.plot(ann.index, ann["gap_mean"].rolling(5, center=True).mean(),
        color=C["blue"], lw=1, alpha=0.6)
ax.set_xlabel("Year")
ax.set_ylabel("Tmax − Tmin (°C)")
ax.set_title("A  Nighttime Recovery Gap")
ax.legend(fontsize=7)
ax.set_xlim(1971, 2025)

# B — Tmin trend separately
ax = axes[1]
sl_tn, ic_tn, r2_tn, p_tn = lin_trend(yrs, ann["tmin_mean"].values)
ax.scatter(ann.index, ann["tmin_mean"], s=20, color=C["purple"], alpha=0.7)
ax.plot(ann.index, sl_tn * yrs + ic_tn, color=C["red"], lw=1.8,
        label=f"Trend: +{sl_tn*10:.3f} °C/decade\n(R²={r2_tn:.2f}, p={p_tn:.3f})")
ax.set_xlabel("Year")
ax.set_ylabel("Annual mean Tmin (°C)")
ax.set_title("B  Nighttime Temperature Trend")
ax.legend(fontsize=7)
ax.set_xlim(1971, 2025)

# C — Warming rates: Tmax vs Tmin (decade bars)
ax = axes[2]
sl_tx, ic_tx, _, _ = lin_trend(yrs, ann["tmax_mean"].values)
rates = {"Tmax": sl_tx * 10, "Tmin": sl_tn * 10,
         "Recovery\nGap": sl_g * 10}
bar_cols = [C["red"], C["purple"], C["blue"]]
bars = ax.bar(rates.keys(), rates.values(), color=bar_cols, alpha=0.8, width=0.5)
for bar, val in zip(bars, rates.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{val:+.3f}", ha="center", va="bottom", fontsize=8)
ax.axhline(0, color=C["black"], lw=0.8)
ax.set_ylabel("Warming rate (°C / decade)")
ax.set_title("C  Warming Rates Compared")
ax.set_ylim(min(rates.values()) - 0.05, max(rates.values()) + 0.08)

fig.tight_layout(w_pad=2.5)
savefig("fig5_recovery_gap.png")

## Figure 6 — Compound Hot-Dry Events

In [ ]:
# ── Figure 6: Compound Events ──────────────────────────────────────────────────
print("Figure 6: Compound events …")

fig, axes = plt.subplots(1, 3, figsize=(7.5, 3.0))

# A — Annual compound event days over time
ax = axes[0]
sl_cp, ic_cp, r2_cp, p_cp = lin_trend(yrs, ann["compound"].values)
ax.bar(ann.index, ann["compound"], color=C["red"], width=0.8, alpha=0.75,
       label="Compound days")
ax.plot(ann.index, ann["compound"].rolling(5, center=True).mean(),
        color=C["black"], lw=1.8, label="5-yr rolling mean")
ax.plot(ann.index, sl_cp * yrs + ic_cp, color=C["orange"], lw=1.5, ls="--",
        label=f"Trend p={p_cp:.3f}")
ax.set_xlabel("Year")
ax.set_ylabel("Compound event days")
ax.set_title("A  Hot + Dry Compound Events")
ax.legend(fontsize=7)
ax.set_xlim(1971, 2025)

# B — Compound vs heatwave days scatter
ax = axes[1]
sc = ax.scatter(ann["hw_days"], ann["compound"],
                c=ann.index, cmap="viridis", s=28, alpha=0.8)
# Regression line
mask = ann["compound"].notna() & ann["hw_days"].notna()
sl_sc, ic_sc, r_sc, p_sc = lin_trend(ann["hw_days"].values, ann["compound"].values)
xr = np.linspace(ann["hw_days"].min(), ann["hw_days"].max(), 50)
ax.plot(xr, sl_sc * xr + ic_sc, color=C["red"], lw=1.5,
        label=f"R={r_sc**0.5:.2f}, p={p_sc:.3f}")
ax.set_xlabel("Heatwave days (Tmax ≥ 36 °C)")
ax.set_ylabel("Compound event days")
ax.set_title("B  Compound vs Heatwave Days")
ax.legend(fontsize=7)
plt.colorbar(sc, ax=ax, label="Year", shrink=0.8, pad=0.02)

# C — Monthly distribution of compound events
ax = axes[2]
monthly_cp = df.groupby(df.index.month)["compound"].sum()
bar_cols_cp = [C["red"] if m in [4, 5] else C["orange"] if m in [3, 6] else C["sky"]
               for m in range(1, 13)]
ax.bar(range(1, 13), monthly_cp, color=bar_cols_cp, width=0.75, alpha=0.85)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax.set_xlabel("Month")
ax.set_ylabel("Total compound days (1972–2024)")
ax.set_title("C  Seasonal Distribution")

fig.tight_layout(w_pad=3.0)
fig.subplots_adjust(top=0.88)
savefig("fig6_compound_events.png")

## Figure 7 — SARIMA Forecast (2025–2029)

In [ ]:
# ── Figure 7: SARIMA Forecast ──────────────────────────────────────────────────
print("Figure 7: SARIMA forecast …")

monthly = df["tmax"].resample("ME").mean().dropna()

print("  Fitting SARIMA(3,1,0)×(1,0,0,12) …")
model = SARIMAX(monthly, order=(3, 1, 0), seasonal_order=(1, 0, 0, 12),
                enforce_stationarity=False, enforce_invertibility=False)
fit = model.fit(disp=False)
print(f"  AIC = {fit.aic:.1f}  BIC = {fit.bic:.1f}")

# Forecast 60 months (2025–2029)
n_months = 60
fc = fit.get_forecast(steps=n_months)
fc_mean = fc.predicted_mean
fc_ci   = fc.conf_int(alpha=0.05)

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.2))

# A — Temperature forecast
ax = axes[0]
hist_plot = monthly.loc["2000":]
ax.plot(hist_plot.index, hist_plot.values, color=C["blue"], lw=1.2,
        alpha=0.85, label="Historical (monthly mean Tmax)")
ax.plot(fc_mean.index, fc_mean.values, color=C["red"], lw=2,
        label="SARIMA forecast (2025–2029)")
ax.fill_between(fc_ci.index, fc_ci.iloc[:, 0], fc_ci.iloc[:, 1],
                color=C["red"], alpha=0.18, label="95% CI")
ax.axvline(monthly.index[-1], color=C["gray"], lw=1, ls=":", alpha=0.7)
ax.set_xlabel("Year")
ax.set_ylabel("Monthly mean Tmax (°C)")
ax.set_title("A  SARIMA Temperature Forecast (2025–2029)")
ax.legend(fontsize=7)

# B — Heatwave day projection (temperature-linked)
ax = axes[1]
# Linear mapping fitted to historical: annual tmax_mean → hw_days
hw_fit_mask = ann["hw_days"].notna() & ann["tmax_mean"].notna()
sl_map, ic_map, r2_map, p_map = lin_trend(ann.loc[hw_fit_mask, "tmax_mean"].values,
                                           ann.loc[hw_fit_mask, "hw_days"].values)
# Annual mean from SARIMA forecast
# The data ends Nov 2024, so the first full forecast year is 2025.
# Resample to calendar year and drop any partial year before the forecast starts.
fc_ann = fc_mean.resample("YE").mean()
# Keep only years ≥ 2025 (first complete forecast year)
fc_ann = fc_ann[fc_ann.index.year >= 2025]
proj_hw = sl_map * fc_ann + ic_map
proj_hw = proj_hw.clip(lower=0)

# Historical
ax.bar(ann.index[-20:], ann["hw_days"].iloc[-20:],
       color=C["sky"], width=0.8, alpha=0.7, label="Historical (last 20 yr)")
proj_years = fc_ann.index.year
ax.bar(proj_years, proj_hw.values, color=C["orange"], width=0.8, alpha=0.85,
       label="Projected (2025–2029)")
ax.axhline(ann["hw_days"].mean(), color=C["gray"], lw=1, ls="--",
           label=f"Historical mean: {ann['hw_days'].mean():.1f} d/yr")
ax.set_xlabel("Year")
ax.set_ylabel("Heatwave days (Tmax ≥ 36 °C)")
ax.set_title("B  Projected Heatwave Days (2025–2029)")
ax.legend(fontsize=7)

# Print forecast table
print("\n  Year-by-year projection:")
print(f"  {'Year':>6}  {'Tmax forecast (°C)':>20}  {'HW days (projected)':>21}")
for yr, tmx, hw in zip(proj_years, fc_ann.values, proj_hw.values):
    print(f"  {yr:>6}  {tmx:>20.2f}  {hw:>21.1f}")
print(f"\n  5-yr mean projected HW days: {proj_hw.mean():.1f}")
print(f"  Historical mean HW days:     {ann['hw_days'].mean():.1f}")

fig.tight_layout(w_pad=2.5)
savefig("fig7_sarima_forecast.png")

## Figure 8 — Climate Drivers Correlation Matrix

In [ ]:
# ── Figure 8: Climate Drivers Correlation ─────────────────────────────────────
print("Figure 8: Correlation matrix …")

corr_cols = {
    "tmax":     "Tmax",
    "tmin":     "Tmin",
    "rh_mean":  "Rel. Humidity",
    "heat_index": "Heat Index",
    "vpd_mean": "VPD",
    "sm_mean":  "Soil Moisture",
    "precip":   "Precipitation",
    "cloud":    "Cloud Cover",
    "sunshine": "Sunshine Dur.",
    "mslp_mean":"MSLP",
}
corr_df = df[list(corr_cols.keys())].rename(columns=corr_cols).dropna()
corr_mat = corr_df.corr(method="pearson")

fig, ax = plt.subplots(figsize=(5.5, 4.8))
import matplotlib.colors as mcolors
cmap = plt.cm.RdBu_r
norm = mcolors.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
im = ax.imshow(corr_mat.values, cmap=cmap, norm=norm, aspect="auto")
n = len(corr_mat)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(corr_mat.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(corr_mat.columns, fontsize=8)
# Annotate cells
for i in range(n):
    for j in range(n):
        val = corr_mat.values[i, j]
        color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=6.5, color=color)
# Add grid lines between cells
for i in range(n + 1):
    ax.axhline(i - 0.5, color="white", lw=0.5)
    ax.axvline(i - 0.5, color="white", lw=0.5)
plt.colorbar(im, ax=ax, label="Pearson r", shrink=0.8, pad=0.02)
ax.set_title("Climate Variable Correlations (Dhaka, 1972–2024)", fontweight="bold")
fig.tight_layout()
savefig("fig8_correlation_matrix.png")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. SUMMARY TABLE — key numbers for manuscript
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("KEY NUMBERS FOR MANUSCRIPT")
print("="*60)

print(f"\nDataset: {len(df):,} days  ({df.index.min().date()} → {df.index.max().date()})")
print(f"Missing Tmax: {df['tmax'].isna().sum()}")
print(f"\nTemperature:")
print(f"  Mean Tmax          : {df['tmax'].mean():.2f} °C")
print(f"  Max Tmax on record : {df['tmax'].max():.1f} °C  ({df['tmax'].idxmax().date()})")
print(f"  Tmax warming trend : {sl:.4f} °C/yr  (= {sl*10:.3f} °C/decade)")
print(f"  Total 1972-2024    : +{sl*52:.2f} °C")
print(f"\nNighttime (Tmin):")
print(f"  Tmin warming trend : {sl_tn:.4f} °C/yr  (= {sl_tn*10:.3f} °C/decade)")
print(f"  Recovery gap trend : {sl_g:.4f} °C/yr  (= {sl_g*10:.3f} °C/decade)")
hw_pct_gap = (sl_g / sl_tn) * 100 if sl_tn != 0 else 0
print(f"  Tmin warming is {'faster' if sl_tn > sl else 'slower'} than Tmax")

print(f"\nHeatwaves (Tmax ≥ 36 °C):")
print(f"  Total days         : {df['hw_day'].sum()}")
print(f"  Historical mean    : {ann['hw_days'].mean():.1f} days/yr")
print(f"  Total events       : {len(hw_ev)}")
print(f"  Median event length: {np.median(hw_ev['length']):.0f} days")
print(f"  Max event length   : {hw_ev['length'].max()} days  ({hw_ev.loc[hw_ev['length'].idxmax(),'start'].date()})")

print(f"\nHeat Index (apparent temperature):")
hi_hw = df[df['hw_day']==1]['heat_index'].dropna()
print(f"  Mean HI on HW days : {hi_hw.mean():.2f} °C  (vs Tmax {df[df['hw_day']==1]['tmax'].mean():.2f} °C)")
print(f"  Mean HI burden (HI-Tmax) on HW days: {(hi_hw - df[df['hw_day']==1]['tmax']).mean():.2f} °C")
print(f"  HI 95th-pct threshold : {HI_95:.2f} °C")
print(f"  Annual mean days above HI 95th pct : {ann['hw_days_hi95'].mean():.1f} days/yr")
print(f"  Trend in extreme HI days (p-value) : {p_95:.4f}  ({'significant' if p_95 < 0.05 else 'not significant'})")

print(f"\nCompound events (hot + dry soil):")
print(f"  Total compound days: {df['compound'].sum()}")
print(f"  Annual mean        : {ann['compound'].mean():.1f} days/yr")
print(f"  Trend (p-value)    : {p_cp:.4f}  ({'significant' if p_cp < 0.05 else 'not significant'} at 0.05)")

print(f"\nDeforestation (2001–2023):")
total_loss = gfw[(gfw["year"] >= 2001) & (gfw["year"] <= 2023)]["tree_loss_ha"].sum()
print(f"  Total tree cover loss: {total_loss/1000:.1f} × 10³ ha")
spear_r, spear_p = stats.spearmanr(
    ann.dropna(subset=["tree_loss_ha"])["tree_loss_ha"],
    ann.dropna(subset=["tree_loss_ha"])["tmax_mean"]
)
print(f"  Spearman ρ (loss vs Tmax): {spear_r:.3f}  (p={spear_p:.3f})")

print(f"\nSARIMA Forecast:")
print(f"  AIC = {fit.aic:.1f},  BIC = {fit.bic:.1f}")
print(f"  5-yr mean projected HW days : {proj_hw.mean():.1f}")
print(f"  Historical mean HW days     : {ann['hw_days'].mean():.1f}")
print(f"  Projected increase          : {((proj_hw.mean()/ann['hw_days'].mean())-1)*100:.0f}%")

print("\nDone. All figures saved to figures/")